In [1]:
import json

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import tqdm

from ase import units
from ase.atoms import Atoms
from ase.build import molecule
from torch_dftd.torch_dftd3_calculator import TorchDFTD3Calculator
from ase.calculators.dftd3 import DFTD3

from cc2cc.utils import gen_mole


class Model(nn.Module):
    """
    Fully connected neural network (dense network)
    """

    def __init__(self, device="cuda", damping="zero", **kwargs):
        super().__init__()

        # device="cuda:0" for fast GPU computation.
        self.calc = TorchDFTD3Calculator(
            device=device,
            dtype=torch.float64,
            xc="b3-lyp",
            damping=damping,
            bidirectional=False,
        )

        if damping == "zero":
            self.param_vector = torch.nn.Parameter(
                torch.tensor(
                    [
                        kwargs.get("rs6", 1.261),
                        kwargs.get("s18", 1.703),
                    ],
                    dtype=torch.float64,
                    device=device,
                )
            )
            self.params = {
                "s6": kwargs.get("s6", 1.0),
                "rs6": self.param_vector[0],
                "s18": self.param_vector[1],
                "rs18": kwargs.get("rs18", 1.0),
                "alp": kwargs.get("alp", 14.0),
            }
        elif damping == "bj":
            self.param_vector = torch.nn.Parameter(
                torch.tensor(
                    [
                        kwargs.get("rs6", 0.3981),
                        kwargs.get("s18", 1.9889),
                        kwargs.get("rs18", 4.4211),
                    ],
                    dtype=torch.float64,
                    device=device,
                )
            )
            self.params = {
                "s6": kwargs.get("s6", 1.0),
                "rs6": self.param_vector[0],
                "s18": self.param_vector[1],
                "rs18": self.param_vector[2],
                "alp": kwargs.get("alp", 14.0),
            }
        self.calc.dftd_module.params = self.params
        self.damping = damping

    def forward(self, batch_dicts):
        self.calc.reset()

        # Calculate the energy using the DFTD3 calculator
        E_disp = self.calc.dftd_module.calc_energy_batch(
            **batch_dicts, damping=self.damping
        )

        return E_disp * units.mol / units.kcal

    def obtain_batch_dicts(self, atoms_list):
        # Calculator.calculate(self, atoms, properties, system_changes)
        input_dicts_list = [self.calc._preprocess_atoms(atoms) for atoms in atoms_list]
        # --- Make batch ---
        n_nodes_list = [d["Z"].shape[0] for d in input_dicts_list]
        shift_index_array = torch.cumsum(torch.tensor([0] + n_nodes_list), dim=0)
        cell_batch = torch.stack(
            [
                (
                    torch.eye(3, device=self.calc.device, dtype=self.calc.dtype)
                    if d["cell"] is None
                    else d["cell"]
                )
                for d in input_dicts_list
            ]
        )

        batch_dicts = dict(
            Z=torch.cat([d["Z"] for d in input_dicts_list], dim=0),  # (n_nodes,)
            pos=torch.cat([d["pos"] for d in input_dicts_list], dim=0),  # (n_nodes,)
            cell=cell_batch,  # (bs, 3, 3)
            pbc=torch.stack([d["pbc"] for d in input_dicts_list]),  # (bs, 3)
            shift_pos=torch.cat(
                [d["shift_pos"] for d in input_dicts_list], dim=0
            ),  # (n_nodes,)
        )
        batch_dicts["edge_index"] = torch.cat(
            [
                d["edge_index"] + shift_index_array[i]
                for i, d in enumerate(input_dicts_list)
            ],
            dim=1,
        )
        batch_dicts["batch"] = torch.cat(
            [
                torch.full((n_nodes,), i, dtype=torch.long, device=self.calc.device)
                for i, n_nodes in enumerate(n_nodes_list)
            ],
            dim=0,
        )
        batch_dicts["batch_edge"] = torch.cat(
            [
                torch.full(
                    (d["edge_index"].shape[1],),
                    i,
                    dtype=torch.long,
                    device=self.calc.device,
                )
                for i, d in enumerate(input_dicts_list)
            ],
            dim=0,
        )

        batch_dicts["pos"].requires_grad_(True)
        return batch_dicts


data = pd.read_csv(
    "/home/dhem/workspace/2025.1/validate/ccdft_cc-pVDZ_atom-1-1424849_gmtkn-cc-pVDZ.csv"
)
data_name_list = (data["name"].str.split("_cc-pVDZ").str[0]).to_numpy()
data_cc_ene = data["cc_ene"].to_numpy() * 627.5094733748099
data_dft_ene = data["dft_ene"].to_numpy() * 627.5094733748099
batch_subset = [
    "W4_11",
    "G21EA",
    "G21IP",
    "DIPCS10",
    "PA26",
    "SIE4x4",
    "ALKBDE10",
    "YBDE18",
    "AL2X6",
    "HEAVYSB11",
    "NBPRC",
    "ALK8",
    "RC21",
    "G2RC",
    "BH76RC",
    "FH51",
    "TAUT15",
    "DC13",
    "MB16_43",
    "DARC",
    "RSE43",
    "BSR36",
    "CDIE20",
    "ISO34",
    # "ISOL24",
    # "C60ISO",
    "PArel",
    "BH76",
    "BHPERI",
    "BHDIV10",
    "INV24",
    "BHROT27",
    "PX13",
    "WCPT18",
    "RG18",
    "ADIM6",
    "S22",
    "S66",
    # "HEAVY28",
    "WATER27",
    "CARBHB12",
    "PNICO23",
    "HAL59",
    "AHB21",
    "CHB6",
    "IL16",
    "IDISP",
    "ICONF",
    "ACONF",
    "Amino20x4",
    "PCONF21",
    "MCONF",
    "SCONF",
    # "UPU23",
    "BUT14DIOL",
]

with open(f"new_dataset/gmtkn-cc-pVDZ.json") as f:
    json_data = json.load(f)

input_batch = {}
name_batch_list = {}
weight_batch_list = {}
mean_absolute_deviation = []
# model = Model(device="cuda", damping="bj")
model = Model(device="cuda", damping="zero")
model.compile(mode="max-autotune-no-cudagraphs")
for name_mol in data_name_list:
    for i_subset in batch_subset:
        if i_subset == "BH76RC":
            i_subset_name = "BH76"
        else:
            i_subset_name = i_subset
        if name_mol.startswith(i_subset_name):
            mol = gen_mole(name_mol, 0, 1, 0, "cc-pVDZ", True, "gmtkn-cc-pVDZ")
            atoms = Atoms(
                symbols=mol.elements, positions=mol.atom_coords() * units.Bohr
            )
            if i_subset not in input_batch:
                input_batch[i_subset] = []
            input_batch[i_subset].append(atoms)
            if i_subset not in name_batch_list:
                name_batch_list[i_subset] = []
            name_batch_list[i_subset].append(name_mol)

for i_subset in batch_subset:
    if i_subset == "BH76RC":
        i_subset_name = "BH76"
    else:
        i_subset_name = i_subset
    reaction_dict = json_data[f"reaction-{i_subset}"]
    name_batch_list[i_subset] = np.array(name_batch_list[i_subset])
    input_batch[i_subset] = model.obtain_batch_dicts(input_batch[i_subset])
    reaction_dict_copy = reaction_dict.copy()
    for i_reaction_name, (i_reaction_keys, i_reaction) in enumerate(
        reaction_dict_copy.items()
    ):
        systems_list = i_reaction["systems"]
        stoichiometry_list = i_reaction["stoichiometry"]

        for i in range(len(systems_list)):
            if i_subset == "BH76RC":
                mole_name = f"{systems_list[i]}"
            else:
                mole_name = f"{i_subset}-{systems_list[i]}"
            stoichiometry = int(stoichiometry_list[i])

            if mole_name in json_data:
                if isinstance(json_data[mole_name], str):
                    mole_name = json_data[mole_name]

            col = np.where(data_name_list == mole_name)[0]
            if col.size != 1:
                print(f"Warning: {mole_name} not found in name_list")
                reaction_dict.pop(i_reaction_keys)
                break
    json_data[f"reaction-{i_subset}"] = reaction_dict

energy_batch_target = {}
for i_subset in batch_subset:
    if i_subset == "BH76RC":
        i_subset_name = "BH76"
    else:
        i_subset_name = i_subset
    reaction_dict = json_data[f"reaction-{i_subset}"]

    energy_batch_target[i_subset] = torch.zeros(
        len(reaction_dict), dtype=torch.float64, device="cpu"
    )
    weight_batch = np.zeros(len(reaction_dict), dtype=np.float64)
    for i_reaction_name, (i_reaction_keys, i_reaction) in enumerate(
        reaction_dict.items()
    ):
        systems_list = i_reaction["systems"]
        stoichiometry_list = i_reaction["stoichiometry"]
        energy_dft = 0

        for i in range(len(systems_list)):
            if i_subset == "BH76RC":
                mole_name = f"{systems_list[i]}"
            else:
                mole_name = f"{i_subset}-{systems_list[i]}"
            stoichiometry = int(stoichiometry_list[i])

            if mole_name in json_data:
                if isinstance(json_data[mole_name], str):
                    mole_name = json_data[mole_name]

            col = np.where(data_name_list == mole_name)[0]
            energy_dft += (data_cc_ene[col[0]] - data_dft_ene[col[0]]) * stoichiometry
            weight_batch[i_reaction_name] += data_cc_ene[col[0]] * stoichiometry
        energy_batch_target[i_subset][i_reaction_name] = energy_dft
    mean_absolute_deviation.extend(np.abs(weight_batch))
    weight_batch_list[i_subset] = 1 / np.mean(np.abs(weight_batch))

print(
    f"mean_absolute_deviation: {np.mean(mean_absolute_deviation) / len(mean_absolute_deviation)}"
)

optimizer = torch.optim.AdamW(model.parameters(), lr=0.0001, weight_decay=1e-5)
loss_function = torch.nn.L1Loss(reduction="sum")
torch.set_printoptions(precision=10)
energy_batch_output = {}
print("start training...")

for epoch in tqdm.tqdm(range(2501)):
    loss_batch = []
    wtmad_2 = 0
    optimizer.zero_grad()
    for i_subset in batch_subset:
        energy = model(input_batch[i_subset])

        reaction_dict = json_data[f"reaction-{i_subset}"]
        energy_batch_output[i_subset] = torch.zeros(
            len(reaction_dict), dtype=torch.float64
        )
        for i_reaction_name, (i_reaction_keys, i_reaction) in enumerate(
            reaction_dict.items()
        ):
            systems_list = i_reaction["systems"]
            stoichiometry_list = i_reaction["stoichiometry"]
            energy_dft = 0

            for i in range(len(systems_list)):
                mole_name = (
                    systems_list[i]
                    if i_subset == "BH76RC"
                    else f"{i_subset}-{systems_list[i]}"
                )
                stoichiometry = int(stoichiometry_list[i])

                if mole_name in json_data:
                    if isinstance(json_data[mole_name], str):
                        mole_name = json_data[mole_name]

                col_disp = np.where(name_batch_list[i_subset] == mole_name)[0]
                if col_disp.size == 1:
                    energy_dft += energy[col_disp[0]] * stoichiometry
                else:
                    print(f"Warning: {mole_name} not found in name_list")
                    break
            energy_batch_output[i_subset][i_reaction_name] = energy_dft
        loss = (
            loss_function(energy_batch_output[i_subset], energy_batch_target[i_subset])
            * weight_batch_list[i_subset]
        )
        loss_batch.append(
            torch.mean(
                torch.abs(energy_batch_output[i_subset] - energy_batch_target[i_subset])
            ).item()
        )
        wtmad_2 += (
            torch.sum(
                torch.abs(energy_batch_output[i_subset] - energy_batch_target[i_subset])
            )
            * weight_batch_list[i_subset]
        ).item()
        # clip the loss to avoid exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(
            f"Epoch: {epoch}, wtmad_2: {wtmad_2 * np.mean(mean_absolute_deviation) / len(mean_absolute_deviation)}, loss: {loss_batch}"
        )

print(f"params_vector {model.params}")

mean_absolute_deviation: 0.051378935483546606
start training...


  0%|          | 1/2501 [00:09<6:24:58,  9.24s/it]

Epoch: 0, wtmad_2: 19.226932291787026, loss: [29.86105500622143, 9.757550138300752, 8.951894574016737, 12.264656864214226, 2.008239726199861, 22.339927819231338, 18.135595643711937, 7.472760193780202, 1.8298921781556086, 5.98415483323616, 2.27423419636448, 2.5590651974408267, 6.029700697565285, 6.417298582695249, 3.536983965379797, 3.2569084306518263, 2.145937033878573, 11.475685719961195, 23.165353859250388, 5.577987780069781, 2.749072878592639, 3.0791274140441627, 1.3475506340978083, 1.6470658482451321, 1.6449560913847527, 9.676054729020095, 6.4139329711172905, 6.535058984711123, 2.228601992386347, 0.7835209465039741, 11.219854402225259, 8.744264182103962, 0.709695661098336, 2.391689657509604, 2.2918677508804777, 2.1020196678537957, 26.402443264768777, 2.8417642561418615, 1.7267543257703541, 2.2472149444794036, 3.2945436106444013, 2.3091202915301414, 4.40999933968787, 5.245958230382064, 0.4932569891361897, 0.2744389838800313, 0.5768979769043694, 1.0994902417874854, 0.7117008296103939

  4%|▍         | 101/2501 [02:02<44:39,  1.12s/it]

Epoch: 100, wtmad_2: 19.06283663615526, loss: [29.85692311833861, 9.757555938694347, 8.951939960774503, 12.26544457772808, 2.0100258045900876, 22.333145421665044, 18.135521289165926, 7.480361334608274, 1.8863779450278975, 5.962063424449416, 2.266956056834907, 2.5687326511203787, 6.011443482071498, 6.409866872165346, 3.5362328648792825, 3.255816668120772, 2.14782931231933, 11.493955280719396, 22.993270819438525, 5.657934762291791, 2.7550712885184607, 3.146931706192385, 1.3490606663282956, 1.6534017305315862, 1.644845865030352, 9.668225453252857, 6.36610945986305, 6.5357981990546365, 2.2320542846948013, 0.7848728669142447, 11.209332354222711, 8.733380828380568, 0.6947287389694883, 2.2966490114208558, 2.2284099290209958, 2.044888745970508, 26.253856964747257, 2.824167363268476, 1.707777342681175, 2.223163457134097, 3.282372855242881, 2.305770573596539, 4.362388965714908, 5.239731328098139, 0.4919101085029753, 0.2623349316267974, 0.573974563431779, 1.0731348519487112, 0.675779574794642, 1.

...

Epoch: 2300, wtmad_2: 16.758022157545206, loss: [29.792083370157954, 9.757323361160118, 8.952412758798788, 12.277552506821944, 2.036189904604873, 22.224967536808478, 18.134064749341192, 7.6040306894210845, 2.768432612588029, 5.6062664563251206, 2.134985591451597, 2.779279361828677, 5.71849831897798, 6.296430247551054, 3.5242702192240696, 3.296977721151204, 2.1668262128500926, 11.786236799118996, 20.175458414612866, 6.885634617666018, 2.842074358823613, 4.376768808665812, 1.3847145313548346, 1.7340862606770318, 1.646644881033501, 9.546038812211531, 5.6274795533859345, 6.503735812126398, 2.308911662495467, 0.80585968985447, 11.128925397232724, 8.567992562375956, 0.46950504711151325, 0.8326406006815485, 1.2048262018016376, 1.118734615199662, 23.644056125132963, 2.5346326679115045, 1.4030113741960832, 1.821442152882052, 3.0949606067438857, 2.2466685413799192, 3.6058924572987507, 5.760833326564317, 0.4737203960761446, 0.11425857546098858, 0.5558772143408979, 0.7734031212096466, 0.4209339675

 96%|█████████▌| 2401/2501 [46:57<02:16,  1.36s/it]

Epoch: 2400, wtmad_2: 16.709552270729798, loss: [29.78985852898981, 9.757306049406797, 8.952423340407531, 12.277891134389108, 2.037106218539945, 22.221638462749496, 18.13400367350546, 7.608468411476025, 2.795917416295705, 5.594204101202669, 2.129832409672398, 2.7877011667551823, 5.708743102977972, 6.2928407124975285, 3.5238747191810305, 3.298914407742028, 2.167080986654127, 11.795345951712493, 20.08830500925423, 6.923354484705194, 2.8447980962611923, 4.419929667491888, 1.3864255541077575, 1.7361259274244636, 1.6471004495944566, 9.542207858964456, 5.605174634987953, 6.501708704095444, 2.3119043621244515, 0.8064411904313618, 11.128533823175655, 8.562765313751397, 0.46358835666117454, 0.7898957107018458, 1.1890065032174137, 1.0902670990316787, 23.56162531992084, 2.5254155100041342, 1.3931100953984135, 1.8079835163849112, 3.0891483105256348, 2.243954840579577, 3.581624384235216, 5.793628213294678, 0.4730359725821552, 0.1134552480520805, 0.5555075477869702, 0.7864582978728394, 0.42278282413

100%|██████████| 2501/2501 [49:08<00:00,  1.18s/it]

Epoch: 2500, wtmad_2: 16.663385643491274, loss: [29.787685505123417, 9.75728890596863, 8.952433542542641, 12.27821814065356, 2.0380047147859255, 22.21840067065959, 18.133943695772608, 7.612815755360909, 2.8226769318542657, 5.58241354606876, 2.124767775760671, 2.795956579386467, 5.6992153299659005, 6.289344699986081, 3.5234892692265523, 3.3008034463420506, 2.1673233136597356, 11.804226824414986, 20.003293860789046, 6.960061235917034, 2.8474523670647534, 4.46221101800517, 1.388115800150961, 1.7380997690830577, 1.6475646211761936, 9.538483307004332, 5.583514989177461, 6.499719870805995, 2.3148364089510056, 0.8069986408275304, 11.128203525953289, 8.557674165750973, 0.45788095342196417, 0.7482481341942934, 1.173590493556858, 1.062535579767208, 23.48144240681564, 2.5164337091208986, 1.3834485342662874, 1.7948313255653066, 3.083492699652802, 2.2412771754074434, 3.55794542072724, 5.825384446874992, 0.4723656553675435, 0.11266609697736793, 0.5551256339540795, 0.7991105211864521, 0.4264083800816

In [2]:
data_dft_bj = []

for name_mol in data_name_list:
    mol = gen_mole(name_mol, 0, 1, 0, "cc-pVDZ", True, "gmtkn-cc-pVDZ")
    atoms = Atoms(
        symbols=mol.elements, positions=mol.atom_coords() * units.Bohr
    )
    energy = model(model.obtain_batch_dicts([atoms]))
    print(f"{name_mol}: {energy.item():.10f} kcal/mol")
    data_dft_bj.append(energy.item() / 627.5094733748099)

data["modified_dft_d3zero"] = data_dft_bj
data.to_csv(
    "/home/dhem/workspace/2025.1/validate/ccdft_cc-pVDZ_atom-1-1424849_gmtkn-cc-pVDZ.csv"
)

W4_11-al: 0.0000000000 kcal/mol
W4_11-b: 0.0000000000 kcal/mol
W4_11-be: 0.0000000000 kcal/mol
W4_11-c: 0.0000000000 kcal/mol
W4_11-cl: 0.0000000000 kcal/mol
W4_11-f: 0.0000000000 kcal/mol
W4_11-h: 0.0000000000 kcal/mol
W4_11-n: 0.0000000000 kcal/mol
W4_11-o: 0.0000000000 kcal/mol
W4_11-p: 0.0000000000 kcal/mol
W4_11-s: 0.0000000000 kcal/mol
W4_11-si: 0.0000000000 kcal/mol
W4_11-alcl: -0.0178873989 kcal/mol
W4_11-alf: -0.0035068805 kcal/mol
W4_11-alh: -0.0026748005 kcal/mol
W4_11-b2: -0.0008629570 kcal/mol
W4_11-be2: -0.1645381521 kcal/mol
W4_11-bf: -0.0006219348 kcal/mol
W4_11-bh: -0.0006457336 kcal/mol
W4_11-bn: -0.0006315803 kcal/mol
W4_11-bn3pi: -0.0008432546 kcal/mol
W4_11-c2: -0.0002454491 kcal/mol
W4_11-cf: -0.0011337204 kcal/mol
W4_11-ch: -0.0002974369 kcal/mol
W4_11-cl2: -0.0119347292 kcal/mol
W4_11-clf: -0.0040158168 kcal/mol
W4_11-clo: -0.0032786780 kcal/mol
W4_11-cn: -0.0003087983 kcal/mol
W4_11-co: -0.0003763976 kcal/mol
W4_11-cs: -0.0020831305 kcal/mol
W4_11-f2: -0.002982

# modified_ai_d3zero
params_vector {'s6': 1.0, 'rs6': tensor(1.5055195208, device='cuda:0', dtype=torch.float64,
       grad_fn=<AsStridedBackward0>), 's18': tensor(1.4496851945, device='cuda:0', dtype=torch.float64,
       grad_fn=<AsStridedBackward0>), 'rs18': 1.0, 'alp': 14.0}